# ARC-AGI-3, offline: an atlas of all 25 games and what the score really means

This notebook is not an agent and it does not solve any game. It is a map of what the competition actually
hands you, and an exact reading of how the score is computed, so you can start in minutes. Everything runs
offline from the bundled wheels and game files: no API key, no internet, no GPU. The one number worth
carrying away is at the end, in the scoring section, because the leaderboard weights levels in a way the
short docs example glosses over.

In [ ]:
import glob, os, sys, subprocess, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt

def find_input():
    for c in ["/kaggle/input/arc-prize-2026-arc-agi-3", "/kaggle/input/arc-prize-2026-arc-agi-3/arc-prize-2026-arc-agi-3"]:
        if os.path.isdir(c):
            return c
    hits = glob.glob("/kaggle/input/**/arc_agi_3_wheels", recursive=True)
    return os.path.dirname(hits[0]) if hits else "/kaggle/input/arc-prize-2026-arc-agi-3"
INPUT = find_input()
WHEELS = glob.glob(os.path.join(INPUT, "**", "arc_agi_3_wheels"), recursive=True)
WHEELS = WHEELS[0] if WHEELS else os.path.join(INPUT, "arc_agi_3_wheels")
# offline install from the competition's own wheel dir; internet is off and stays off
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", WHEELS,
                "arc_agi", "arcengine"], check=False)
import arc_agi, arcengine
print("arc_agi", getattr(arc_agi, "__version__", "?"), "| arcengine", getattr(arcengine, "__version__", "?"))
print("input:", INPUT)

## The 25 games at a glance

Each game ships as a small Python engine plus a `metadata.json` that names it, tags it, and gives the
baseline action budget per level (the number of moves a competent human needs). The table below is read
straight from those files. The tag families matter: `keyboard` games take a few directional actions,
`keyboard_click` games also expose a click at an x, y cell.

In [ ]:
META = sorted(glob.glob(os.path.join(INPUT, "**", "environment_files", "*", "*", "metadata.json"), recursive=True))
rows = []
for mp in META:
    m = json.load(open(mp))
    ba = m.get("baseline_actions") or []
    rows.append(dict(game_id=m.get("game_id", os.path.basename(os.path.dirname(os.path.dirname(mp))))[:4],
                     title=m.get("title"), tags=",".join(m.get("tags") or []),
                     n_levels=len(ba), human_budget=int(sum(ba)), meta_path=mp))
GAMES = pd.DataFrame(rows).sort_values("game_id").reset_index(drop=True)
print(f"{len(GAMES)} games bundled")
print(GAMES[["game_id", "title", "tags", "n_levels", "human_budget"]].to_string(index=False))

In [ ]:
from arc_agi import LocalEnvironmentWrapper
from arc_agi.models import EnvironmentInfo
from arc_agi.rendering import frame_to_rgb_array, COLOR_MAP
from arcengine import GameAction, GameState
import logging
_log = logging.getLogger("atlas"); logging.basicConfig(level=logging.ERROR)

def make_env(meta_path):
    # returns (meta, EnvironmentInfo, env) or (meta, info, None) if the game did not load
    m = json.load(open(meta_path))
    info = EnvironmentInfo(game_id=m["game_id"], title=m.get("title"), default_fps=m.get("default_fps"),
                           tags=m.get("tags"), baseline_actions=m.get("baseline_actions"),
                           local_dir=os.path.dirname(meta_path))
    try:
        env = LocalEnvironmentWrapper(info, _log, scorecard_id="atlas")
        _ = env.observation_space.frame  # reset happened in __init__
        return m, info, env
    except Exception as e:
        print("  did not load:", info.game_id, type(e).__name__, str(e)[:70])
        return m, info, None

loaded = 0
for mp in META:
    _, _, env = make_env(mp)
    loaded += env is not None
print(f"{loaded} of {len(META)} games loaded and reset cleanly offline")

## The atlas

One panel per game, its opening screen rendered with the competition's own 16 colour palette. This is the
whole roster in a single view, the direct analog of a task gallery, so you can see the visual variety of
what you are up against. The palette key underneath decodes the grid values.

In [ ]:
import math
n = len(META); cols = 5; rows_ = math.ceil(n / cols)
fig, axes = plt.subplots(rows_, cols, figsize=(cols*2.5, rows_*2.5))
axes = np.array(axes).reshape(-1)
for ax in axes: ax.axis("off")
for i, mp in enumerate(META):
    m, info, env = make_env(mp)
    ax = axes[i]
    if env is None:
        ax.text(0.5, 0.5, info.game_id + "\n(did not load)", ha="center", va="center", fontsize=8); continue
    fr = env.observation_space
    rgb = frame_to_rgb_array(0, np.array(fr.frame[0]), scale=6)
    ax.imshow(rgb, interpolation="nearest")
    ax.set_title(f"{info.game_id[:4]}  L{len(info.baseline_actions or [])}  a{len(fr.available_actions)}", fontsize=8)
plt.suptitle("ARC-AGI-3: the opening frame of all 25 games", fontsize=12, y=1.0)
plt.tight_layout(); plt.show()

# palette key
fig, ax = plt.subplots(figsize=(9, 1.1))
for k, v in sorted(COLOR_MAP.items())[:16]:
    ax.add_patch(plt.Rectangle((k, 0), 1, 1, color=v[:7]))  # COLOR_MAP holds #RRGGBBAA hex; #RRGGBB is enough
    ax.text(k+0.5, -0.35, str(k), ha="center", fontsize=8)
ax.set_xlim(0, 16); ax.set_ylim(-0.6, 1); ax.axis("off"); ax.set_title("the 16 colour palette, index under each swatch", fontsize=10)
plt.tight_layout(); plt.show()

## Watch a game respond

To make the observation and action cycle concrete, here is one navigation game taking a dozen random simple
actions. This is not skilled play, it just shows that `step` mutates the grid and that the game exposes its
own progress on screen. Read it left to right.

In [ ]:
import random
nav = next((mp for mp in META if "ls20" in mp), META[0])
m, info, env = make_env(nav)
random.seed(0)
simple = [GameAction.ACTION1, GameAction.ACTION2, GameAction.ACTION3, GameAction.ACTION4]
strip = [np.array(env.observation_space.frame[0]).copy()]
for _ in range(11):
    a = random.choice([s for s in simple if s.value in (env.observation_space.available_actions or [s.value for s in simple])] or simple)
    r = env.step(a, data={}, reasoning={})
    strip.append(np.array(r.frame[0]).copy())
fig, axes = plt.subplots(1, len(strip), figsize=(len(strip)*1.5, 1.7))
for i, (ax, g) in enumerate(zip(axes, strip)):
    ax.imshow(frame_to_rgb_array(0, g, scale=4), interpolation="nearest"); ax.axis("off")
    ax.set_title("start" if i == 0 else f"+{i}", fontsize=8)
plt.suptitle(f"{info.game_id[:4]} under 11 random simple actions (final state {r.state})", fontsize=10)
plt.tight_layout(); plt.show()

## What actions each game gives you

The engine only ever offers legal actions. Simple actions 1 to 5 and 7 take no argument; action 6 is a
click at an x, y cell. The count per game below tells you at a glance which games are pure navigation and
which expose the click.

In [ ]:
av = []
for mp in META:
    m, info, env = make_env(mp)
    if env is None: continue
    acts = env.observation_space.available_actions or []
    av.append(dict(game_id=info.game_id[:4], n_actions=len(acts), actions=str(sorted(acts)),
                   has_click=6 in acts))
AV = pd.DataFrame(av).sort_values("game_id")
print(AV.to_string(index=False))
print(f"\ngames exposing the click (action 6): {int(AV.has_click.sum())} of {len(AV)}")

## The score, exactly, from the source

This is the part people get wrong. Per completed level the score is `min((baseline / actions)^2 * 100, 115)`,
and an uncompleted level scores 0. A game's score is the level-number-weighted mean of its level scores,
then hard-capped at `(weight of completed levels / weight of all levels) * 100`. The final leaderboard is
the plain mean over games. The weighting and the cap are why finishing the easy early levels is worth far
less than the short docs example suggests. Below the hand formula is cross-checked against the competition's
own `EnvironmentScoreCalculator` so you can trust it.

In [ ]:
from arc_agi.scorecard import EnvironmentScoreCalculator

def hand_game_score(baselines, actions, completed):
    tot = w = maxw = 0.0
    for i, (b, a, c) in enumerate(zip(baselines, actions, completed), start=1):
        s = min((b / a) ** 2 * 100, 115.0) if (c and a > 0) else 0.0
        tot += s * i; w += i
        if s > 0: maxw += i
    return min(tot / w, maxw / w * 100) if w else 0.0

def official_game_score(baselines, actions, completed):
    calc = EnvironmentScoreCalculator(id="x")
    for i, (b, a, c) in enumerate(zip(baselines, actions, completed), start=1):
        calc.add_level(level_index=i, completed=bool(c), actions_taken=int(a), baseline_actions=int(b))
    return calc.to_score().score

BASE = [55, 8, 41, 21, 23, 23]   # cd82's six level baselines
scenarios = {
    "match human on all 6": (BASE, [True]*6),
    "double the actions on all 6": ([2*b for b in BASE], [True]*6),
    "first 4 of 6 perfect, quit": (BASE, [True, True, True, True, False, False]),
}
print(f"{'scenario':32s} {'hand':>8s} {'official':>9s}")
for name, (acts, comp) in scenarios.items():
    h = hand_game_score(BASE, acts, comp); o = official_game_score(BASE, acts, comp)
    print(f"{name:32s} {h:8.2f} {o:9.2f}   {'match' if abs(h-o) < 1e-6 else 'MISMATCH'}")
print("\nnote: first 4 of 6 perfect scores 47.62, not 66.7; unfinished late levels are punished hard.")

ks = np.linspace(1, 3, 100)
fig, ax = plt.subplots(figsize=(6.4, 3.0))
ax.plot(ks, np.minimum((1/ks)**2*100, 115), color="#B23A2E")
ax.axhline(100, color="#14212B", lw=0.7, ls=":"); ax.axhline(25, color="#4E6B7A", lw=0.7, ls=":")
ax.annotate("match human -> 100", (1.0, 100), (1.15, 78), fontsize=8, arrowprops=dict(arrowstyle="->"))
ax.annotate("double the actions -> 25", (2.0, 25), (2.05, 45), fontsize=8, arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("actions taken / human baseline"); ax.set_ylabel("level score")
ax.set_title("Level score decays with the square of your action overshoot")
plt.tight_layout(); plt.show()

## A forkable skeleton

Everything you need to start is below: load a game, loop while it is unfinished and you are under budget,
read the frame and state, stop on a win or a game over. It plays randomly on purpose. Replace `choose_action`
with your solver and you have an agent. No API key, no network.

In [ ]:
# map an available action id back to its GameAction member (the enum's int constructor is unreliable here)
VAL2ACT = {m.value: m for m in GameAction}

def choose_action(frame, available, rng):
    # your solver goes here; this picks a random legal simple action
    simple_legal = [a for a in (available or []) if a in (1, 2, 3, 4, 5, 7)] or [1]
    return VAL2ACT.get(int(rng.choice(simple_legal)), GameAction.ACTION1)

def run_one(meta_path, max_actions=40, seed=0):
    m, info, env = make_env(meta_path)
    if env is None: return None
    rng = np.random.default_rng(seed)
    fr = env.observation_space; taken = 0
    while fr.state in (GameState.NOT_PLAYED, GameState.NOT_FINISHED) and taken < max_actions:
        a = choose_action(fr.frame[0], fr.available_actions, rng)
        fr = env.step(a, data={}, reasoning={}); taken += 1
    return dict(game=info.game_id[:4], actions=taken, final_state=str(fr.state),
                levels_completed=getattr(fr, "levels_completed", None))

demo = run_one(next((mp for mp in META if "ls20" in mp), META[0]))
print("random skeleton run:", demo)

## What to take away

This is a map and an exact ruler, not a solver. If you remember one thing, remember the scoring cap: a game
is scored by a level-number-weighted mean and then capped by the fraction of level weight you actually
finished, so a late unfinished level costs far more than the flat docs example implies. Build for finishing
deep, not for shaving actions on the levels you already clear. Fork the skeleton above and drop in your
`choose_action`.